# Ch 00 · Chapter Template

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/master/dpdk-research/mujoco-warp-course/notebooks/_template.ipynb)

Companion notebook for the chapter page `chapters/_template.html` ([page](../chapters/_template.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
# (not a rendering notebook: MUJOCO_GL left unset)
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

## 1. One launch, many worlds
Each example below is the same file as on the chapter page.

In [ ]:
# Example 0.1 - ex01_ball_drop.py
"""Example T.1 - drop a ball in NWORLD parallel worlds (template chapter demo).

Runs on CPU (small budget) and on a Colab GPU (big budget) without changes.
Run: python ex01_ball_drop.py [--dump-json PATH]
  --dump-json PATH writes decimated per-world heights for the page's replay player (Rule D).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

BALL_DROP = """
<mujoco>
  <option timestep="0.002"/>
  <worldbody>
    <geom type="plane" size="5 5 0.1"/>
    <body pos="0 0 1">
      <freejoint/>
      <geom type="sphere" size="0.1" mass="1"/>
    </body>
  </worldbody>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()  # parse_known_args: also works inside Jupyter

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 1000 if ON_GPU else 250

mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=4, njmax=16)

# start every world at a different height: qpos[:, 2] is the ball's z
qpos = np.tile(mjm.qpos0, (NWORLD, 1)).astype(np.float32)
qpos[:, 2] = np.linspace(0.5, 1.5, NWORLD)
d.qpos.assign(qpos)

frames = []
for i in range(NSTEP):
    mjw.step(m, d)
    if args.dump_json and i % 5 == 0:   # decimate: every 5th step (reads to host: demo only)
        frames.append([round(float(v), 3) for v in d.qpos.numpy()[:, 2]])
wp.synchronize()

z = d.qpos.numpy()[:, 2]
print("device:", wp.get_device())
print("qpos shape:", d.qpos.shape)
print(f"t = {NSTEP * mjm.opt.timestep:.2f} s, z first 4 worlds:", z[:4])
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    dt = 5 * float(mjm.opt.timestep)
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": NWORLD, "dt": dt, "kind": "height", "zmin": 0.0, "zmax": 1.6,
                   "frames": frames}, f, separators=(",", ":"))
    print("wrote", args.dump_json, len(frames), "frames")

## 2. A custom Warp kernel over MJWarp state

In [ ]:
# Example 0.2 - ex02_energy_kernel.py
"""Example T.2 - a custom Warp kernel that reads MJWarp's batched state (template chapter demo).

One thread per world computes the ball's energy from d.qpos and d.qvel.
Run: python ex02_energy_kernel.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

BALL_DROP = """
<mujoco>
  <option timestep="0.002"/>
  <worldbody>
    <geom type="plane" size="5 5 0.1"/>
    <body pos="0 0 1">
      <freejoint/>
      <geom type="sphere" size="0.1" mass="1"/>
    </body>
  </worldbody>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 4
NSTEP = 100


@wp.kernel
def ball_energy(qpos: wp.array2d(dtype=float), qvel: wp.array2d(dtype=float),
                mass: float, g: float, energy: wp.array(dtype=float)):
    worldid = wp.tid()                      # one thread per world
    v = wp.vec3(qvel[worldid, 0], qvel[worldid, 1], qvel[worldid, 2])
    energy[worldid] = 0.5 * mass * wp.dot(v, v) + mass * g * qpos[worldid, 2]


mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=4, njmax=16)
energy = wp.zeros(NWORLD, dtype=float)

for _ in range(NSTEP):
    mjw.step(m, d)
wp.launch(ball_energy, dim=d.nworld, inputs=[d.qpos, d.qvel, 1.0, 9.81], outputs=[energy])
print("energy per world [J]:", energy.numpy()[:4])
print("expected at rest height 1 m:", round(1.0 * 9.81 * 1.0, 4))

## Exercise 0.1: landing time
Predict when a ball dropped from 1 m (radius 0.1 m) touches the floor.

In [ ]:
t_pred = ...  # your formula
# assert abs(t_sim - t_pred) / t_pred < 0.05

In [ ]:
#@title Exercise 0.1: solution (double-click to show the code)
t_pred = (2 * 0.9 / 9.81) ** 0.5
print(f"predicted landing time: {t_pred:.3f} s")